In [ ]:
!pip install sentence-transformers chromadb pypdf pandas openpyxl openai


In [ ]:
!pip install streamlit

In [ ]:
!pip install python-dotenv

In [ ]:
%%writefile .env
OPENROUTER_API_KEY="your API Key"

Overwriting .env


In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import os



class Settings:

    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

    MODEL_NAME = os.environ.get(
        "OPENROUTER_MODEL",
        "openai/gpt-oss-20b:free"
    )

    EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

    CHROMA_COLLECTION_NAME = "academic_research"

    DEFAULT_TOP_K = 5

    CHUNK_SIZE = 500
    CHUNK_OVERLAP = 50

    SUPPORTED_EXTENSIONS = [
        ".pdf",
        ".txt",
        ".csv",
        ".xlsx"
    ]

    LOG_FILE = "pipeline.log"


settings = Settings()


if settings.OPENROUTER_API_KEY:
    print("✅ OPENROUTER_API_KEY detected — real LLM enabled.")
else:
    print(
        "⚠️ No OPENROUTER_API_KEY set — "
        "the system will use a mock LLM response."
    )

✅ OPENROUTER_API_KEY detected — real LLM enabled.


## 3. Pipeline Logging

Create a logging component that records important events during document validation, processing, embedding, retrieval, and LLM generation.

In [ ]:
import logging


class PipelineLogger:

    def __init__(self, log_file: str = settings.LOG_FILE):

        logging.basicConfig(
            filename=log_file,
            level=logging.INFO,
            format="%(asctime)s | %(levelname)s | %(message)s"
        )

        self.logger = logging.getLogger("ResearchRAG")

    def info(self, message: str):
        print(f"ℹ️ {message}")
        self.logger.info(message)

    def warning(self, message: str):
        print(f"⚠️ {message}")
        self.logger.warning(message)

    def error(self, message: str):
        print(f"❌ {message}")
        self.logger.error(message)

In [ ]:
logger = PipelineLogger()

logger.info("Research RAG pipeline initialized.")

ℹ️ Research RAG pipeline initialized.


## 4. Document Validation

Validate incoming documents before they are added to the RAG knowledge base.

The validation checks whether the file exists, uses a supported format, and is not empty.

In [ ]:
from openai import OpenAI

In [ ]:
class DocumentValidationResult:

    def __init__(self):

        self.valid = True
        self.errors = []

    def add_error(self, message):

        self.valid = False
        self.errors.append(message)

    def summary(self):

        return {
            "valid": self.valid,
            "errors": self.errors
        }

In [ ]:
import os
import hashlib


class DocumentValidator:

    def __init__(self):
        self.supported_extensions = settings.SUPPORTED_EXTENSIONS

    def validate_file(self, file_path: str) -> dict:

        result = {
            "file": file_path,
            "valid": True,
            "errors": []
        }

        # 1. File exists
        if not os.path.exists(file_path):
            result["valid"] = False
            result["errors"].append("File does not exist.")
            return result

        # 2. File extension
        extension = os.path.splitext(file_path)[1].lower()

        if extension not in self.supported_extensions:
            result["valid"] = False
            result["errors"].append(
                f"Unsupported file format: {extension}"
            )

        # 3. Empty file
        if os.path.getsize(file_path) == 0:
            result["valid"] = False
            result["errors"].append("File is empty.")

        return result

## 5. Content Quality Validation

After validating the file itself, validate the extracted content to ensure that the document contains enough readable text to be processed by the RAG pipeline.

In [ ]:
class DocumentValidator:

    def __init__(self):

        self.supported_extensions = [
            ".pdf"
        ]

        self.min_text_length = 100

    def calculate_hash(self, file_path):

        sha256 = hashlib.sha256()

        with open(
            file_path,
            "rb"
        ) as file:

            for block in iter(
                lambda: file.read(4096),
                b""
            ):

                sha256.update(block)

        return sha256.hexdigest()

    def validate_file(
        self,
        file_path,
        metadata=None
    ):

        result = DocumentValidationResult()

        # ==================================
        # 1. File exists
        # ==================================

        if not os.path.exists(file_path):

            result.add_error(
                "File does not exist."
            )

            return result

        # ==================================
        # 2. Supported format
        # ==================================

        extension = os.path.splitext(
            file_path
        )[1].lower()

        if extension not in self.supported_extensions:

            result.add_error(
                f"Unsupported format: {extension}"
            )

            return result

        # ==================================
        # 3. Empty file
        # ==================================

        if os.path.getsize(
            file_path
        ) == 0:

            result.add_error(
                "Document is empty."
            )

            return result

        # ==================================
        # 4. Validate and extract PDF
        # ==================================

        extracted_text = ""

        try:

            reader = PdfReader(
                file_path
            )

            if len(reader.pages) == 0:

                result.add_error(
                    "PDF contains no pages."
                )

                return result

            for page in reader.pages:

                page_text = (
                    page.extract_text()
                )

                if page_text:

                    extracted_text += (
                        page_text + "\n"
                    )

        except Exception as e:

            result.add_error(
                f"Corrupted PDF: {str(e)}"
            )

            return result

        # ==================================
        # 5. Extractable text validation
        # ==================================

        clean_text = (
            extracted_text.strip()
        )

        if len(clean_text) < self.min_text_length:

            result.add_error(
                "Insufficient extractable text."
            )

        return result

## 6. Document Text Extraction

Extract readable text from supported documents. PDF files are processed using PyPDF, while TXT, CSV, and Excel files are also supported.

In [ ]:
from pypdf import PdfReader
import pandas as pd


class DocumentExtractor:

    @staticmethod
    def extract(file_path: str) -> str:

        extension = os.path.splitext(file_path)[1].lower()

        # PDF
        if extension == ".pdf":

            reader = PdfReader(file_path)

            pages = []

            for page in reader.pages:

                text = page.extract_text()

                if text:
                    pages.append(text)

            return "\n".join(pages)

        # TXT
        elif extension == ".txt":

            with open(
                file_path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as file:

                return file.read()

        # CSV
        elif extension == ".csv":

            df = pd.read_csv(file_path)

            return df.to_string(index=False)

        # Excel
        elif extension == ".xlsx":

            df = pd.read_excel(file_path)

            return df.to_string(index=False)

        else:

            raise ValueError(
                f"Unsupported file type: {extension}"
            )

In [ ]:
class ContentValidator:

    MIN_TEXT_LENGTH = 100

    @staticmethod
    def validate_text(text: str) -> dict:

        result = {
            "valid": True,
            "errors": []
        }

        if not text or not text.strip():

            result["valid"] = False
            result["errors"].append(
                "No extractable text found."
            )

            return result

        if len(text.strip()) < ContentValidator.MIN_TEXT_LENGTH:

            result["valid"] = False
            result["errors"].append(
                "Document contains insufficient extractable text."
            )

        return result

## 7. Text Cleaning

Clean extracted research text by removing unnecessary whitespace and invalid characters before creating chunks.

In [ ]:
import re


class TextCleaner:

    @staticmethod
    def clean(text: str) -> str:

        text = text.replace("\x00", " ")

        text = re.sub(
            r"\s+",
            " ",
            text
        )

        return text.strip()

## 8. Text Chunking

Split each research document into smaller overlapping chunks. These chunks will later be converted into embeddings and stored in the vector database.

In [ ]:
class TextChunker:

    def __init__(
        self,
        chunk_size: int = settings.CHUNK_SIZE,
        overlap: int = settings.CHUNK_OVERLAP
    ):

        self.chunk_size = chunk_size
        self.overlap = overlap

    def split(self, text: str) -> list:

        chunks = []

        start = 0

        while start < len(text):

            end = start + self.chunk_size

            chunk = text[start:end].strip()

            if chunk:
                chunks.append(chunk)

            start = end - self.overlap

        return chunks

## 9. Generate Text Embeddings

Use a Sentence Transformer model to convert research text chunks and user questions into numerical vectors that can be compared semantically.

In [ ]:
from sentence_transformers import SentenceTransformer


class EmbeddingModel:

    def __init__(
        self,
        model_name: str = settings.EMBEDDING_MODEL_NAME
    ):

        print(
            f"⏳ Loading embedding model '{model_name}'..."
        )

        self.model = SentenceTransformer(model_name)

        print("✅ Embedding model ready.")

    def encode(self, text: str):

        embedding = self.model.encode(text)

        return embedding.tolist()

    def encode_many(self, texts: list):

        embeddings = self.model.encode(texts)

        return embeddings.tolist()

## 10. ChromaDB Vector Database

Store document chunks, their embeddings, and metadata in ChromaDB so that relevant research content can later be retrieved using semantic search.

In [ ]:
class ChromaManager:

    def __init__(
        self,
        collection_name=(
            settings.CHROMA_COLLECTION_NAME
        )
    ):

        self.client = (
            chromadb.Client()
        )

        self.collection = (
            self.client
            .get_or_create_collection(
                name=collection_name
            )
        )

    def add_chunks(
        self,
        ids,
        documents,
        embeddings,
        metadatas
    ):

        self.collection.add(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas
        )

    def search(
        self,
        embedding,
        top_k=settings.DEFAULT_TOP_K
    ):

        return self.collection.query(
            query_embeddings=[embedding],
            n_results=top_k
        )

## 11. Semantic Research Retrieval

Convert the user's research question into an embedding and retrieve the most semantically relevant research chunks from ChromaDB.

In [ ]:
class ResearchRetriever:

    def __init__(
        self,
        embedding_model,
        chroma_manager
    ):

        self.embedding_model = (
            embedding_model
        )

        self.chroma_manager = (
            chroma_manager
        )

    def retrieve(
        self,
        question,
        top_k=settings.DEFAULT_TOP_K
    ):

        question_embedding = (
            self.embedding_model.encode(
                question
            )
        )

        return (
            self.chroma_manager.search(
                question_embedding,
                top_k
            )
        )

## 12. RAG Prompt

Define the prompt used by the LLM to answer questions using only the retrieved academic research context.

In [ ]:
RESEARCH_PROMPT = """
You are an Academic Research & Literature
Discovery Assistant.

Answer the user's question using ONLY the
provided research context.

Do not invent information.

If the answer cannot be found in the provided
context, say:

"The information was not found in the
research knowledge base."

User Question:
{question}

Research Context:
{context}

Instructions:

1. Give a clear and concise answer.
2. Use only the provided research context.
3. Mention the source paper when possible.
4. Do not invent citations.
5. Do not make unsupported claims.
"""

## 13. LLM Client

Connect the RAG pipeline to an LLM through OpenRouter. If no API key is available, use a mock response so that the rest of the pipeline can still be tested.

In [ ]:
class LLMClient:

    def __init__(
        self,
        api_key=(
            settings.OPENROUTER_API_KEY
        ),
        model_name=(
            settings.MODEL_NAME
        )
    ):

        self.api_key = api_key

        self.model_name = model_name

        if self.api_key:

            self.client = OpenAI(
                base_url=(
                    "https://openrouter.ai/api/v1"
                ),
                api_key=self.api_key
            )

            print(
                "Real LLM enabled."
            )

        else:

            self.client = None

            print(
                "Mock LLM enabled."
            )

    def generate(
        self,
        prompt
    ):

        if not self.client:

            return (
                "[MOCK RESPONSE]\n"
                "No API key was configured."
            )

        try:

            response = (
                self.client
                .chat
                .completions
                .create(
                    model=self.model_name,
                    messages=[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    temperature=0.2
                )
            )

            return (
                response
                .choices[0]
                .message
                .content
            )

        except Exception as e:

            logger.error(
                f"LLM call failed: {e}"
            )

            return (
                "[MOCK RESPONSE]\n"
                "The LLM request failed."
            )

## 14. Research Assistant Service

Combine document validation, extraction, cleaning, chunking, embedding, vector storage, retrieval, and LLM generation into one end-to-end service.

In [ ]:
import chromadb
test_client = chromadb.Client()

print("✅ ChromaDB is working correctly.")

✅ ChromaDB is working correctly.


In [ ]:
class ResearchAssistantService:

    def __init__(self):

        # Embedding model
        self.embedding_model = EmbeddingModel()

        # Document validation
        self.validator = DocumentValidator()

        # ChromaDB
        self.chroma_manager = ChromaManager()

        # LLM / RAG
        self.llm = LLMClient()

        # Track indexed documents
        self.indexed_hashes = set()

        print("✅ Research Assistant Service initialized.")

    # =========================================================
    # DOCUMENT INGESTION
    # =========================================================

    def ingest_document(
        self,
        file_path,
        metadata=None
    ):
        """
        Validate, extract, clean, chunk, embed,
        and store a research document in ChromaDB.
        """

        errors = []

        try:

            # -------------------------------------------------
            # 1. Validate
            # -------------------------------------------------

            validation_result = (
                self.validator.validate_file(
                    file_path,
                    metadata=metadata
                )
            )

            if not validation_result.valid:

                return {
                    "indexed": False,
                    "errors": validation_result.errors,
                    "chunks": 0
                }

            # -------------------------------------------------
            # 2. Duplicate detection
            # -------------------------------------------------

            document_hash = (
                self.validator.calculate_hash(
                    file_path
                )
            )

            if document_hash in self.indexed_hashes:

                return {
                    "indexed": False,
                    "errors": [
                        "Duplicate document detected."
                    ],
                    "chunks": 0
                }

            # -------------------------------------------------
            # 3. Extract text
            # -------------------------------------------------

            text = self._extract_text(
                file_path
            )

            if not text or not text.strip():

                return {
                    "indexed": False,
                    "errors": [
                        "No extractable text found."
                    ],
                    "chunks": 0
                }

            # -------------------------------------------------
            # 4. Clean text
            # -------------------------------------------------

            cleaned_text = self._clean_text(
                text
            )

            # -------------------------------------------------
            # 5. Split into chunks
            # -------------------------------------------------

            chunks = self._create_chunks(
                cleaned_text
            )

            if not chunks:

                return {
                    "indexed": False,
                    "errors": [
                        "No text chunks were created."
                    ],
                    "chunks": 0
                }

            # -------------------------------------------------
            # 6. Generate embeddings
            # -------------------------------------------------

            embeddings = []

            for chunk in chunks:

                embedding = (
                    self.embedding_model.encode(
                        chunk
                    )
                )

                embeddings.append(
                    embedding
                )

            # -------------------------------------------------
            # 7. Prepare ChromaDB data
            # -------------------------------------------------

            document_name = os.path.basename(
                file_path
            )

            document_id = (
                document_hash[:16]
            )

            ids = []

            metadatas = []

            for i in range(
                len(chunks)
            ):

                ids.append(
                    f"{document_id}_chunk_{i}"
                )

                chunk_metadata = {
                    "source": document_name,
                    "document_id": document_id,
                    "chunk_id": i
                }

                if metadata:

                    chunk_metadata.update(
                        metadata
                    )

                metadatas.append(
                    chunk_metadata
                )

            # -------------------------------------------------
            # 8. Store in ChromaDB
            # -------------------------------------------------

            self.chroma_manager.collection.add(
                ids=ids,
                documents=chunks,
                embeddings=embeddings,
                metadatas=metadatas
            )

            # -------------------------------------------------
            # 9. Mark document as indexed
            # -------------------------------------------------

            self.indexed_hashes.add(
                document_hash
            )

            return {
                "indexed": True,
                "errors": [],
                "chunks": len(chunks),
                "document_id": document_id
            }

        except Exception as e:

            return {
                "indexed": False,
                "errors": [
                    f"{type(e).__name__}: {str(e)}"
                ],
                "chunks": 0
            }

    # =========================================================
    # TEXT EXTRACTION
    # =========================================================

    def _extract_text(
        self,
        file_path
    ):

        extension = os.path.splitext(
            file_path
        )[1].lower()

        # PDF
        if extension == ".pdf":

            reader = PdfReader(
                file_path
            )

            pages = []

            for page in reader.pages:

                page_text = (
                    page.extract_text()
                )

                if page_text:

                    pages.append(
                        page_text
                    )

            return "\n".join(
                pages
            )

        # TXT
        elif extension == ".txt":

            with open(
                file_path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                return f.read()

        # CSV
        elif extension == ".csv":

            df = pd.read_csv(
                file_path
            )

            return df.to_string(
                index=False
            )

        # Excel
        elif extension == ".xlsx":

            df = pd.read_excel(
                file_path
            )

            return df.to_string(
                index=False
            )

        else:

            raise ValueError(
                f"Unsupported file format: {extension}"
            )

    # =========================================================
    # TEXT CLEANING
    # =========================================================

    def _clean_text(
        self,
        text
    ):

        text = re.sub(
            r"\s+",
            " ",
            text
        )

        return text.strip()

    # =========================================================
    # CHUNKING
    # =========================================================

    def _create_chunks(
        self,
        text,
        chunk_size=500,
        overlap=50
    ):

        words = text.split()

        chunks = []

        start = 0

        while start < len(words):

            end = start + chunk_size

            chunk = " ".join(
                words[start:end]
            )

            if chunk.strip():

                chunks.append(
                    chunk
                )

            start = end - overlap

        return chunks

    # =========================================================
    # QUESTION ANSWERING
    # =========================================================

    def ask(
        self,
        question,
        top_k=5
    ):

        # Generate question embedding
        question_embedding = (
            self.embedding_model.encode(
                question
            )
        )

        # Retrieve relevant chunks
        results = (
            self.chroma_manager.collection.query(
                query_embeddings=[
                    question_embedding
                ],
                n_results=top_k
            )
        )

        documents = (
            results.get(
                "documents",
                [[]]
            )[0]
        )

        metadatas = (
            results.get(
                "metadatas",
                [[]]
            )[0]
        )

        if not documents:

            return {
                "answer": (
                    "I could not find relevant "
                    "research content."
                ),
                "sources": []
            }

        # Combine retrieved chunks
        context = "\n\n".join(
            documents
        )

        # Sources
        sources = []

        for metadata in metadatas:

            if metadata:

                source = metadata.get(
                    "source",
                    "Unknown"
                )

                if source not in sources:

                    sources.append(
                        source
                    )

        # RAG prompt
        prompt = f"""
You are an academic research assistant.

Answer the user's question using ONLY
the research context provided below.

If the answer cannot be found in the
provided research context, say that the
information is not available in the
indexed research documents.

Research Context:
{context}

Question:
{question}

Answer:
"""

        # Generate answer
        answer = self.llm.generate(
            prompt
        )

        return {
            "answer": answer,
            "sources": sources
        }

## 15. Initialize the Research Assistant

Initialize the complete research assistant service.

The service combines document validation, content extraction, text cleaning, chunking, embedding generation, ChromaDB storage, semantic retrieval, and LLM-based answer generation.

In [ ]:
assistant = ResearchAssistantService()

print("✅ Research Assistant is ready.")

⏳ Loading embedding model 'sentence-transformers/all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model ready.
Real LLM enabled.
✅ Research Assistant Service initialized.
✅ Research Assistant is ready.


## 16. Gradio User Interface

Use Gradio to create a simple and user-friendly interface for uploading research documents and asking questions about the validated research knowledge base.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

## 17. Document Upload and Validation

Create the function used by the Gradio interface to receive an uploaded research document.

The uploaded document is first validated. Only documents that pass validation continue to the indexing stage.

In [ ]:
def upload_and_process(file):

    if file is None:

        return (
            "❌ No file uploaded.",
            "Please upload a research PDF."
        )

    try:

        file_path = str(file)

        print(
            f"Processing file: {file_path}"
        )

        # ==================================
        # Metadata
        # ==================================
        #
        # No manual title or author input.
        # The document itself will be used as
        # the source.
        #

        metadata = {
            "source": os.path.basename(
                file_path
            )
        }

        # ==================================
        # VALIDATION
        # ==================================

        validation = (
            assistant.validator.validate_file(
                file_path,
                metadata=None
            )
        )

        if not validation.valid:

            errors = "\n".join(
                [
                    f"• {error}"
                    for error in validation.errors
                ]
            )

            return (
                "❌ DOCUMENT REJECTED\n\n"
                + errors,

                "❌ Document was NOT indexed."
            )

        # ==================================
        # INDEXING
        # ==================================

        result = (
            assistant.ingest_document(
                file_path,
                metadata=metadata
            )
        )

        if not result["indexed"]:

            errors = "\n".join(
                result.get(
                    "errors",
                    ["Unknown indexing error."]
                )
            )

            return (
                "❌ INDEXING FAILED\n\n"
                + errors,

                "❌ Document was NOT stored."
            )

        return (
            "✅ DOCUMENT ACCEPTED\n\n"
            "The document passed validation.",

            (
                "✅ INDEXING SUCCESSFUL\n\n"
                f"File: {os.path.basename(file_path)}\n"
                f"Chunks: {result['chunks']}\n"
                "Stored in ChromaDB."
            )
        )

    except Exception as e:

        return (
            "❌ SYSTEM ERROR\n\n"
            f"{type(e).__name__}: {str(e)}",

            "The document could not be processed."
        )

## 18. Research Question Answering

Create the function that receives a user's research question, retrieves relevant content from ChromaDB, and generates an answer using the LLM.

In [ ]:
def ask_question(question):

    # Check whether the user entered a question
    if question is None or not question.strip():

        return (
            "❌ Please enter a research question.",
            ""
        )

    try:

        # Send the question to the Research Assistant
        result = assistant.ask(
            question
        )

        # Get generated answer
        answer = result.get(
            "answer",
            "No answer was generated."
        )

        # Get sources
        sources = result.get(
            "sources",
            []
        )

        # Format sources
        if sources:

            sources_text = "\n".join(
                [
                    f"• {source}"
                    for source in sources
                ]
            )

        else:

            sources_text = (
                "No sources found."
            )

        return (
            answer,
            sources_text
        )

    except Exception as e:

        return (
            "❌ SYSTEM ERROR\n\n"
            f"{type(e).__name__}: {str(e)}",

            ""
        )

## 19. Build the Gradio Interface

Create the user interface for:

1. Uploading a research document.
2. Providing paper metadata.
3. Validating and indexing the document.
4. Asking research questions.
5. Displaying the generated answer.
6. Displaying the retrieved sources.

In [ ]:
with gr.Blocks(
    title="Academic Research Assistant"
) as demo:

    gr.Markdown(
        """
        # 📚 Academic Research & Literature Discovery Assistant

        Upload a research document, validate it,
        and ask questions about the indexed research.
        """
    )

    # ======================================
    # DOCUMENT UPLOAD
    # ======================================

    gr.Markdown(
        "## 📄 Upload Research Document"
    )

    file_input = gr.File(
        label="Research PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    process_button = gr.Button(
        "Validate & Index Document",
        variant="primary"
    )

    validation_output = gr.Textbox(
        label="Validation Result",
        lines=8
    )

    indexing_output = gr.Textbox(
        label="Indexing Information",
        lines=8
    )

    process_button.click(
        fn=upload_and_process,
        inputs=file_input,
        outputs=[
            validation_output,
            indexing_output
        ]
    )

    # ======================================
    # QUESTION ANSWERING
    # ======================================

    gr.Markdown(
        "## 🔎 Ask a Research Question"
    )

    question_input = gr.Textbox(
        label="Research Question",
        placeholder=(
            "Example: What are the main findings "
            "of the research paper?"
        ),
        lines=3
    )

    ask_button = gr.Button(
        "Ask Question",
        variant="primary"
    )

    answer_output = gr.Textbox(
        label="Answer",
        lines=10
    )

    sources_output = gr.Textbox(
        label="Sources",
        lines=5
    )

    ask_button.click(
        fn=ask_question,
        inputs=question_input,
        outputs=[
            answer_output,
            sources_output
        ]
    )

print("✅ Gradio interface created.")

✅ Gradio interface created.


## 20. Launch the Gradio Interface

Launch the user-friendly Gradio interface.

Research documents will be uploaded through this interface rather than directly through the Colab notebook.

In [ ]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8540384368345ea7a3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Processing file: /tmp/gradio/019d538c8f693f7e704770c8f82be505b4e1f808e0ee75086538a53f0b080a7f/valid_research_paper.pdf


Processing file: /tmp/gradio/f9e67f59ad7aae41af73993c4f86a129625b1f5f88429bfdcbb8dff18ccc8ef2/invalid_research_paper.pdf
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8540384368345ea7a3.gradio.live
